<a href="https://colab.research.google.com/github/MarrissaSyokwaa/datascienceprojects/blob/main/HUGGING_FACE_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load Pre-trained BERT (Base Uncased)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 2. Define 10 Sentence Pairs with Ground Truth (1=Similar, 0=Not Similar)
sentence_pairs = [
    ("The weather is nice today.", "It is a sunny day.", 1),
    ("I love programming in Python.", "Python is a great language for coding.", 1),
    ("The bank was closed.", "He sat by the river bank.", 0),
    ("The cat sat on the mat.", "A feline rested on the rug.", 1),
    ("The stock market crashed.", "I am cooking pasta for dinner.", 0),
    # Additional 5 Pairs
    ("The bright star is visible.", "The movie star signed autographs.", 0),
    ("I need to recharge my phone.", "The battery is almost dead.", 1),
    ("He is a fast runner.", "He moves with great speed.", 1),
    ("The apple was sweet.", "He works at the Apple store.", 0),
    ("The teacher explained the lesson.", "The instructor gave a lecture.", 1)
]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# 1. Install necessary library (Run this in a separate cell if not installed)
# !pip install transformers torch scikit-learn

import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 2. Load the Model and Tokenizer
# This is what triggered your download warnings
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 3. Define the 10 Sentence Pairs and Ground Truth (GT)
# GT 1: Similar meaning | GT 0: Different meaning
pairs = [
    ("The weather is nice today.", "It is a sunny day.", 1),
    ("I love programming in Python.", "Python is a great language for coding.", 1),
    ("The bank was closed.", "He sat by the river bank.", 0), # Polysemy check
    ("The cat sat on the mat.", "A feline rested on the rug.", 1),
    ("The stock market crashed.", "I am cooking pasta for dinner.", 0),
    # My 5 additional pairs:
    ("The bright star is visible.", "The movie star signed autographs.", 0), # Context check
    ("I need to recharge my phone.", "The battery is almost dead.", 1),
    ("He is a fast runner.", "He moves with great speed.", 1),
    ("The apple was sweet.", "He works at the Apple store.", 0),
    ("The teacher explained the lesson.", "The instructor gave a lecture.", 1)
]

# 4. Function to extract the [CLS] embedding
def get_cls_embedding(sentence):
    # Tokenize and move to tensors
    inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Extract the [CLS] token (first token at index 0)
    return outputs.last_hidden_state[:, 0, :].numpy()

# 5. Compute Similarity and Accuracy
predictions = []
ground_truths = [p[2] for p in pairs]

print(f"{'Sentence Pair':<70} | {'Sim':<6} | {'Pred':<4} | {'GT'}")
print("-" * 90)

for s1, s2, gt in pairs:
    emb1 = get_cls_embedding(s1)
    emb2 = get_cls_embedding(s2)

    # Cosine Similarity
    sim = cosine_similarity(emb1, emb2)[0][0]

    # Threshold logic
    pred = 1 if sim > 0.7 else 0
    predictions.append(pred)

    pair_text = f"{s1[:33]} vs {s2[:33]}"
    print(f"{pair_text:<70} | {sim:.3f} | {pred:<4} | {gt}")

# 6. Final Accuracy
from sklearn.metrics import accuracy_score
acc = accuracy_score(ground_truths, predictions)
print(f"\nFinal Assignment Accuracy: {acc * 100}%")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence Pair                                                          | Sim    | Pred | GT
------------------------------------------------------------------------------------------
The weather is nice today. vs It is a sunny day.                       | 0.944 | 1    | 1
I love programming in Python. vs Python is a great language for co     | 0.906 | 1    | 1
The bank was closed. vs He sat by the river bank.                      | 0.891 | 1    | 0
The cat sat on the mat. vs A feline rested on the rug.                 | 0.936 | 1    | 1
The stock market crashed. vs I am cooking pasta for dinner.            | 0.743 | 1    | 0
The bright star is visible. vs The movie star signed autographs.       | 0.765 | 1    | 0
I need to recharge my phone. vs The battery is almost dead.            | 0.943 | 1    | 1
He is a fast runner. vs He moves with great speed.                     | 0.872 | 1    | 1
The apple was sweet. vs He works at the Apple store.                   | 0.819 | 1    | 0
The tea